# Multi-head Attention

Reimplementation of the multi-head self-attention (MSA) layer used in the Transformer encoder of [An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale](https://arxiv.org/abs/2010.11929) (Dosovitskiy et al., 2020), using only basic PyTorch layers. The attention itself follows [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017).

![Multi-head Attention](../assets/multi_head_attention.gif)

In this notebook I implement scaled dot-product attention and combine several attention heads into the $\text{MSA}$ block from Eq. 2, applied to the embedded patches produced in the previous notebook.

**Contents**
1. Setup
2. Data
3. Scaled dot-product attention
4. Multi-head attention


## 1. Setup

In [1]:
import torch
import torch.nn as nn

In [2]:
def set_seed(seed: int = 42):
    """Seeds PyTorch's CPU and CUDA random number generators for reproducible runs.

    Args:
        seed: Seed value passed to the generators.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

## 2. Data

The input is $\mathbf{z}_0$ from Eq. 1, the class token and embedded patches of a single image saved in the patch embedding notebook, with shape $(B, N+1, D)$.

In [3]:
embedded_patches_example = torch.load("patch_embedding_sample.pt")

In [4]:
print(embedded_patches_example[0, :10])
print(embedded_patches_example.shape)

tensor([[ 0.0109, -0.0094, -0.0243,  ..., -0.0091,  0.0090,  0.0024],
        [-0.4790,  0.1003, -0.1863,  ...,  0.4345, -0.2351,  0.2104],
        [-0.4672,  0.1485, -0.1737,  ...,  0.4147, -0.3149,  0.1660],
        ...,
        [-0.4990,  0.1077, -0.1480,  ...,  0.4059, -0.2472,  0.3080],
        [-0.3080,  0.1658, -0.1072,  ...,  0.2522, -0.1388,  0.1091],
        [-0.3206, -0.0556, -0.1652,  ...,  0.4020, -0.1186,  0.0959]],
       grad_fn=<SliceBackward0>)
torch.Size([1, 257, 768])


## 3. Scaled dot-product attention

Each head projects the tokens into queries, keys and values, $Q = \mathbf{z}W_q$, $K = \mathbf{z}W_k$, $V = \mathbf{z}W_v$, and computes

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

With $h$ heads, each head works in $d_k = d_v = D / h$ dimensions.

### 3.1 Step by step

In [5]:
d_model = embedded_patches_example.shape[2]
h = 12
d_k, d_v = d_model // h, d_model // h
print(f"Dimensions of key are: {d_k}")
print(f"Dimensions of value are: {d_v}")

Dimensions of key are: 64
Dimensions of value are: 64


In [6]:
set_seed()

W_q = torch.randn(size=(d_model, d_k)) * 0.1
W_k = torch.randn(size=(d_model, d_k)) * 0.1
W_v = torch.randn(size=(d_model, d_v)) * 0.1

print(f"Weights for query: {W_q.shape}")
print(f"Weights for key: {W_k.shape}")
print(f"Weights for value: {W_v.shape}")

Weights for query: torch.Size([768, 64])
Weights for key: torch.Size([768, 64])
Weights for value: torch.Size([768, 64])


In [7]:
Q = torch.matmul(embedded_patches_example, W_q)
K = torch.matmul(embedded_patches_example, W_k)
V = torch.matmul(embedded_patches_example, W_v)

print(f"Query shape: {Q.shape}")
print(f"Key shape: {K.shape}")
print(f"Value shape: {V.shape}")

Query shape: torch.Size([1, 257, 64])
Key shape: torch.Size([1, 257, 64])
Value shape: torch.Size([1, 257, 64])


The attention weights have shape $(N+1) \times (N+1)$, one row per query token and one column per key token. Softmax over the last dimension makes each row sum to 1.

In [8]:
attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k))

softmax = nn.Softmax(dim=-1)
attention_weights = softmax(attention_scores)

attention_output = torch.matmul(attention_weights, V)

print(f"Attention output: {attention_output[0, :10]}")
print(f"Attention scores shape: {attention_scores.shape}")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"Attention output shape: {attention_output.shape}")

Attention output: tensor([[-1.0120, -1.0014, -0.4140, -0.8210,  0.1574,  0.3147,  1.1753,  0.1305,
         -1.2012, -0.4453, -0.8400, -0.2502, -0.9207, -1.0451, -0.2104, -1.1082,
         -0.6562,  0.6131,  1.1742, -0.8071, -0.9024,  0.4233, -0.4574, -0.3190,
         -1.0543,  0.5171,  0.4310,  0.8943,  0.6948, -0.1616, -1.3125,  0.9634,
         -0.3427,  0.9012,  0.0332, -0.5729,  0.1216,  1.0735, -0.7043,  0.2221,
         -0.8290,  0.4027, -0.5798, -0.6073,  0.2630, -3.5798,  1.2360, -0.0906,
          0.9794, -1.7380, -0.6366,  1.0742,  0.7951, -0.2011, -0.8283, -0.2431,
         -0.0988,  0.2043, -0.5966,  0.3493, -0.2717,  0.7510, -0.6879, -1.0850],
        [-0.9763, -0.9626, -0.3915, -0.7911,  0.1500,  0.3004,  1.1355,  0.1210,
         -1.1615, -0.4240, -0.8129, -0.2368, -0.8769, -1.0128, -0.2010, -1.0641,
         -0.6268,  0.5927,  1.1346, -0.7723, -0.8747,  0.4102, -0.4412, -0.3121,
         -1.0209,  0.4967,  0.4137,  0.8672,  0.6721, -0.1594, -1.2704,  0.9353,
         

### 3.2 `ScaleDotProductAttention` module

In [9]:
class ScaleDotProductAttention(nn.Module):
    """Single attention head computing softmax(QK^T / sqrt(d_k))V over a sequence of tokens.

    Args:
        d_model: Dimension D of the input tokens.
        d_k: Dimension of the queries and keys.
        d_v: Dimension of the values.
    """

    def __init__(self, d_model: int, d_k: int, d_v: int):
        super().__init__()
        self.d_k = d_k
        self.d_v = d_v
        self.W_q = nn.Linear(d_model, d_k)
        self.W_k = nn.Linear(d_model, d_k)
        self.W_v = nn.Linear(d_model, d_v)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Computes single-head self-attention.

        Args:
            x: Tensor of shape (B, N, d_model).

        Returns:
            Tensor of shape (B, N, d_v).
        """
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1)
        output = torch.matmul(attention_weights, V)

        return output

## 4. Multi-head attention

All $h$ heads read the same input in parallel, their outputs are concatenated and mixed by the output projection $W_O \in \mathbb{R}^{hd_v \times D}$:

$$\text{MSA}(\mathbf{z}) = [\text{head}_1; \dots; \text{head}_h]\,W_O$$

The output has the same shape as the input, so it can be added back to $\mathbf{z}$ in the residual connection of Eq. 2.

### 4.1 Step by step

In [10]:
set_seed()

attention_heads = nn.ModuleList([ScaleDotProductAttention(d_model, d_k, d_v) for _ in range(h)])

head_outputs = [head(embedded_patches_example) for head in attention_heads]
multiple_head_output = torch.cat(head_outputs, dim=-1)

W_o = nn.Linear(h * d_v, d_model)
multiple_head_output = W_o(multiple_head_output)

print(f"Multiple head output shape: {multiple_head_output.shape}")
print(f"Multiple head output: {multiple_head_output[0, :10]}")

Multiple head output shape: torch.Size([1, 257, 768])
Multiple head output: tensor([[ 0.1069, -0.0028,  0.0323,  ...,  0.0292,  0.0843, -0.1053],
        [ 0.1067, -0.0029,  0.0323,  ...,  0.0291,  0.0842, -0.1052],
        [ 0.1067, -0.0029,  0.0323,  ...,  0.0291,  0.0842, -0.1052],
        ...,
        [ 0.1068, -0.0029,  0.0323,  ...,  0.0292,  0.0842, -0.1052],
        [ 0.1067, -0.0029,  0.0323,  ...,  0.0291,  0.0842, -0.1052],
        [ 0.1068, -0.0029,  0.0323,  ...,  0.0291,  0.0842, -0.1053]],
       grad_fn=<SliceBackward0>)


### 4.2 `MultiHeadAttention` module

In [11]:
class MultiHeadAttention(nn.Module):
    """Runs h attention heads in parallel, concatenates their outputs and projects them back to d_model.

    Args:
        d_model: Dimension D of the input tokens.
        d_k: Dimension of the queries and keys of each head.
        d_v: Dimension of the values of each head.
        h: Number of attention heads.
    """

    def __init__(self, d_model: int, d_k: int, d_v: int, h: int):
        super().__init__()
        self.h = h
        self.d_k = d_k
        self.d_v = d_v
        self.attention_heads = nn.ModuleList([ScaleDotProductAttention(d_model, d_k, d_v) for _ in range(h)])
        self.W_o = nn.Linear(h * d_v, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Computes multi-head self-attention.

        Args:
            x: Tensor of shape (B, N, d_model).

        Returns:
            Tensor of shape (B, N, d_model).
        """
        head_outputs = [head(x) for head in self.attention_heads]
        multiple_head_output = torch.cat(head_outputs, dim=-1)
        output = self.W_o(multiple_head_output)
        return output

### 4.3 Comparison with `nn.MultiheadAttention`

With identical weights, the custom module should produce the same output as PyTorch's implementation.

In [12]:
set_seed()

custom_multi_head_attention = MultiHeadAttention(d_model, d_k, d_v, h)

pytorch_multi_head_attention = nn.MultiheadAttention(embed_dim=d_model, num_heads=h, batch_first=True)

# Both modules are randomly initialized, so outputs only match after sharing weights.
# nn.MultiheadAttention packs Q, K, V of all heads into one (3 * d_model, d_model) matrix,
# ordered [Q_head1..Q_headh, K_head1..K_headh, V_head1..V_headh]
heads = custom_multi_head_attention.attention_heads
with torch.no_grad():
    for i, name in enumerate(["W_q", "W_k", "W_v"]):
        rows = slice(i * d_model, (i + 1) * d_model)
        pytorch_multi_head_attention.in_proj_weight[rows] = torch.cat([getattr(head, name).weight for head in heads])
        pytorch_multi_head_attention.in_proj_bias[rows] = torch.cat([getattr(head, name).bias for head in heads])
    pytorch_multi_head_attention.out_proj.weight.copy_(custom_multi_head_attention.W_o.weight)
    pytorch_multi_head_attention.out_proj.bias.copy_(custom_multi_head_attention.W_o.bias)

x = embedded_patches_example
custom_multi_head_output = custom_multi_head_attention(x)
pytorch_multi_head_output = pytorch_multi_head_attention(x, x, x)[0]

torch.allclose(custom_multi_head_output, pytorch_multi_head_output, atol=1e-6)

True